# Somo 11 - Itifaki ya Mwakala kwa Mwakala (A2A)


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Je, Itifaki ya A2A ni nini?

The **Agent-to-Agent (A2A) protocol** ni standardi iliyo wazi inayowawezesha maajenti wa AI kuwasiliana, kugundua kila mmoja, na kushirikiana — hata wanapojengwa kwenye mifumo tofauti au kuendeshwa na huduma tofauti.

Misingi muhimu:

- **Discovery** – Maajenti huchapisha *Agent Card* inayoelezea uwezo wao, ikifanya iwe rahisi kwa maajenti wengine (au maratibu) kupata mtaalamu sahihi kwa kazi.
- **Message Passing** – Maajenti hubadilishana ujumbe uliopangwa kupitia itifaki ya pamoja, hivyo ombi kutoka kwa ajenti mmoja linaweza kueleweka na kutimizwa na mwingine bila kujali utekelezaji wake wa ndani.
- **Task Lifecycle** – A2A inabainisha hali kama *submitted*, *working*, *completed*, na *failed*, ikimpa maratibu uwazi kamili juu ya jinsi kazi iliyotumwa inavyoendelea.

Katika somo hili tunaiga ushirikiano wa mtindo wa A2A kwa kuunganisha maajenti wa kusafiri watatu waliobobea katika mtiririko wa kazi ambapo kila ajenti huchangia utaalamu wake na kupeleka matokeo kwa ajenti inayofuata.


## Kuunda mawakala maalum wa kusafiri


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Ushirikiano wa Wakala Wengi kupitia Mtiririko wa Kazi

Tunawaunganisha wakala watatu katika mtiririko wa kazi wa mfululizo unaoiga kusafirisha ujumbe kati ya wakala (A2A):

1. **CurrencyExchangeAgent** anapokea ombi la mtumiaji na kutoa mwongozo kuhusu sarafu.
2. **ActivityPlannerAgent** anapokea muktadha ulioboreshwa na kuongeza mapendekezo ya shughuli.
3. **TravelManagerAgent** anachanganya pembejeo zote mbili kuwa muhtasari wa mwisho wa safari.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuelewa A2A katika Uzalishaji

Katika mazingira ya uzalishaji itifaki ya A2A inafungua matukio yenye nguvu ya juu-zaid ya huduma:

| Capability | Description |
|---|---|
| **Uunganishaji kati ya mifumo tofauti** | Wakala aliyejengwa kwa kutumia mfumo mmoja anaweza kuwasilisha kazi kwa wakala aliyejengwa kwa kutumia mfumo mwingine wowote unaoendana na A2A, kuwezesha uingilianaji wa kweli kati ya mashirika. |
| **Mipaka ya huduma** | Wakala wanaweza kuishi katika microservices tofauti, mikoa tofauti ya wingu, au hata mashirika tofauti huku wakishirikiana bila mshono. |
| **Ugunduzi unaoendelea** | Mratibu anaweza kuuliza rejista ya Kadi ya Wakala wakati wa utekelezaji ili kupata mtaalamu anayefaa zaidi kwa kazi ndogo iliyotengwa. |
| **Utiririko & arifa za kusukuma** | A2A inaunga mkono Server-Sent Events (SSE) kwa sasisho za maendeleo kwa wakati halisi na arifa za kusukuma kwa kazi zinazoendelea kwa muda mrefu. |

Mtiririko wa kazi tulioujenga hapo juu ni toleo lililorahishwa, la ndani ya mchakato, la mfano huu. Katika utekelezaji halisi
kila wakala angeweka wazi anwani ya HTTP, kuchapisha Kadi ya Wakala, na kuwasiliana
kupitia itifaki ya A2A JSON-RPC.


## Muhtasari

Katika somo hili ulijifunza:

1. **Itifaki ya A2A ni nini** — kiwango wazi cha kugundua, kutuma ujumbe, na usimamizi wa kazi kati ya maajenti.
2. **Jinsi ya kuunda maajenti maalum** — ajenti wa Kubadilisha Sarafu, ajenti wa Mpangaji wa Shughuli, na mpangiliaji wa Meneja wa Safari.
3. **Jinsi ya kuunganisha maajenti katika mtiririko wa kazi** — kutumia `WorkflowBuilder` kuiga mfululizo
   wa upitishaji ujumbe kati ya maajenti.
4. **Jinsi A2A inavyofanya kazi katika mazingira ya uzalishaji** — kuwezesha ushirikiano kati ya mifumo tofauti na huduma tofauti
   kwa ugundaji unaobadilika na masasisho yanayotiririka.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Tamko la kutokuwajibika:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya utafsiri ya AI Co-op Translator (https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuwa sahihi, tafadhali kumbuka kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au kutokuwa sahihi. Nyaraka ya asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo cha kuaminika. Kwa taarifa muhimu, inashauriwa kutumia tafsiri ya kitaalamu iliyofanywa na binadamu. Hatuwajibiki kwa kutokuelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
